In [ ]:
import os
import json
import numpy as np
import cv2
import random
from pathlib import Path

In [ ]:
def setup_folders():
    folders = ['image_masks', 'image_previews']
    #masks 已標記的黑白圖 #previews紅色遮罩圖, 用來檢查

    for f in folders:
        Path(f).mkdir(parents=True, exist_ok=True)

    print(f"Folder has been created: {folders}")

In [ ]:
def convert_labelme_json_to_masks(sample_rate=0.2):
    input_folder = 'preprocessing_output'
    output_folder = 'masks_output'
    preview_folder = 'preview_output'
    folders = [input_folder, output_folder, preview_folder]

    for folder in folders:
        if not os.path.exists(folder):
            print(f"{folder} not exist")
            return
    
    json_extensions = ['.json']

    json_files = [os.path.join(input_folder, f) for f in os.listdir(input_folder)
                  #listdir僅回傳檔名, f只接收檔名, 因後面code需要這邊用os.path.join(input_folder, f)強制回傳路徑(最高層為程式所在目錄的下一層)
                  if os.path.splitext(f)[1].lower() in json_extensions]

    if not json_files:
        print("json file not exist")
        return
    
    print(f"total .json files : {len(json_files)}")

    sample_count = max(1, int(len(json_files) * sample_rate)) #max(1, x)這邊用於避免x<1
    preview_targets = random.sample(json_files, sample_count) #random.sample隨機取出不重複的目標 #random.sample(清單, 數量)

    for json_path in json_files:
        try: #try對應except, 出問題時跳至except
            with open(json_path, 'r', encoding='utf-8') as f:
            #with...as f完成讀取後自動關閉檔案 #open(檔案路徑, 讀檔方式, 解碼方式)
                data = json.load(f)
                #.load()以()中的方式讀取並解析檔案
                #data會是labelme產生的dictionary, 格式如下
                # {
                # "version": "5.0.1",
                # "flags": {},
                # "shapes": [ ... ],
                # "imagePath": "cell_01.jpg",
                # "imageData": null,
                # "imageHeight": 1024,
                # "imageWidth": 1280
                # }